# Data Wrangling and Feature Engineering for Machine Learning

This notebook prepares a leakage-aware machine-learning table from the raw wastewater dataset.

Main decisions:

- Parse `sampling_date` and coerce water-quality columns to numeric values.
- Create final COD removal targets from influent and final effluent COD values.
- Use only influent and anaerobic effluent measurements as predictors.
- Exclude raw final effluent columns from model features because they are part of, or measured after, the final outcome.
- Engineer process features that describe influent composition, anaerobic-stage removal, solids ratios, and date/order effects.
- Save a new CSV for downstream machine-learning experiments.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load Raw Data


In [2]:
RAW_PATH = Path("water_quality_raw.csv")
OUTPUT_PATH = Path("water_quality_ml_ready.csv")

df_raw = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()


Raw shape: (27, 25)


,sampling_date,inf_pH,inf_cond,inf_CODt,inf_CODs,inf_MLSS,inf_MLVSS,inf_TKN,inf_NH4N,ana_pH,ana_cond,ana_CODt,ana_CODs,ana_MLSS,ana_MLVSS,ana_TKN,ana_NH4N,eff_pH,eff_cond,eff_CODt,eff_CODs,eff_MLSS,eff_MLVSS,eff_TKN,eff_NH4N
0,01/12/2025,NaN,NaN,2695,2152,308.00,276.00,112.00,25.00,NaN,NaN,652,150,500.00,467.00,141.00,100.00,NaN,NaN,180,145,4.00,4.00,13,6.00
1,03/12/2025,3.95,NaN,4772,4349,255.00,215.00,258.00,50.00,7.41,NaN,999,306,630.00,440.00,159.00,100.00,7.56,NaN,147,139,5.00,5.00,13,6.00
2,05/12/2025,4.48,NaN,3935,3531,450.00,385.00,197.00,44.00,7.20,NaN,997,326,670.00,500.00,148.00,92.00,7.53,NaN,114,105,6.00,0.00,12,1.00
3,08/12/2025,4.99,NaN,2402,2255,233.00,197.00,111.00,98.00,7.55,NaN,612,321,305.00,250.00,154.00,112.00,7.56,NaN,135,98,37.00,17.00,13,6.00
4,10/12/2025,NaN,NaN,2993,1712,290.00,195.00,64.00,15.00,NaN,NaN,940,354,393.00,273.00,149.00,103.00,NaN,NaN,119,85,12.00,0.00,16,8.00


In [3]:
df = df_raw.copy()
df["sampling_date"] = pd.to_datetime(df["sampling_date"], dayfirst=True, errors="coerce")

numeric_cols = df.columns.drop("sampling_date")
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

df = df.sort_values("sampling_date").reset_index(drop=True)

print(f"Cleaned shape before feature engineering: {df.shape}")
print(f"Date range: {df['sampling_date'].min().date()} to {df['sampling_date'].max().date()}")
df.dtypes


Cleaned shape before feature engineering: (27, 25)
Date range: 2025-12-01 to 2026-04-27


sampling_date    datetime64[us]
inf_pH                  float64
inf_cond                float64
inf_CODt                  int64
inf_CODs                  int64
inf_MLSS                float64
inf_MLVSS               float64
inf_TKN                 float64
inf_NH4N                float64
ana_pH                  float64
ana_cond                float64
ana_CODt                  int64
ana_CODs                  int64
ana_MLSS                float64
ana_MLVSS               float64
ana_TKN                 float64
ana_NH4N                float64
eff_pH                  float64
eff_cond                float64
eff_CODt                  int64
eff_CODs                  int64
eff_MLSS                float64
eff_MLVSS               float64
eff_TKN                   int64
eff_NH4N                float64
dtype: object

## 2. Helper Functions


In [4]:
def safe_ratio(numerator, denominator):
    # Return numerator / denominator while avoiding divide-by-zero artifacts.
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def removal_pct(upstream, downstream):
    # Return removal percentage from upstream to downstream concentration.
    upstream = upstream.replace(0, np.nan)
    return (upstream - downstream) / upstream * 100


## 3. Create Prediction Targets


In [5]:
# Final treatment performance targets. These are outcomes, not predictors.
df["target_final_CODt_removal_pct"] = removal_pct(df["inf_CODt"], df["eff_CODt"])
df["target_final_CODs_removal_pct"] = removal_pct(df["inf_CODs"], df["eff_CODs"])

target_cols = [
    "target_final_CODt_removal_pct",
    "target_final_CODs_removal_pct",
]

df[target_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
target_final_CODt_removal_pct,27.00,93.81,4.59,80.22,92.48,96.02,97.01,97.69
target_final_CODs_removal_pct,27.00,93.66,3.43,84.78,91.87,94.42,96.80,97.56


## 4. Engineer Upstream Process Features


In [6]:
# Optional Features
# Date/order features. These can capture slow process drift over the sampling campaign.
# df["sample_index"] = np.arange(len(df))
# df["sampling_month"] = df["sampling_date"].dt.month
# df["sampling_dayofyear"] = df["sampling_date"].dt.dayofyear
# df["days_since_first_sample"] = (df["sampling_date"] - df["sampling_date"].min()).dt.days

# Influent composition features.
df["inf_CODs_to_CODt_ratio"] = safe_ratio(df["inf_CODs"], df["inf_CODt"])
df["inf_MLVSS_to_MLSS_ratio"] = safe_ratio(df["inf_MLVSS"], df["inf_MLSS"])
df["inf_NH4N_to_TKN_ratio"] = safe_ratio(df["inf_NH4N"], df["inf_TKN"])
df["inf_CODt_to_TKN_ratio"] = safe_ratio(df["inf_CODt"], df["inf_TKN"])
df["inf_CODs_to_TKN_ratio"] = safe_ratio(df["inf_CODs"], df["inf_TKN"])

# Anaerobic effluent composition features.
df["ana_CODs_to_CODt_ratio"] = safe_ratio(df["ana_CODs"], df["ana_CODt"])
df["ana_MLVSS_to_MLSS_ratio"] = safe_ratio(df["ana_MLVSS"], df["ana_MLSS"])
df["ana_NH4N_to_TKN_ratio"] = safe_ratio(df["ana_NH4N"], df["ana_TKN"])
df["ana_CODt_to_TKN_ratio"] = safe_ratio(df["ana_CODt"], df["ana_TKN"])
df["ana_CODs_to_TKN_ratio"] = safe_ratio(df["ana_CODs"], df["ana_TKN"])

# Anaerobic-stage removal/transformation features. These are valid predictors only if anaerobic effluent data are available before final prediction.
df["ana_CODt_removal_pct"] = removal_pct(df["inf_CODt"], df["ana_CODt"])
df["ana_CODs_removal_pct"] = removal_pct(df["inf_CODs"], df["ana_CODs"])
df["ana_TKN_change_pct"] = removal_pct(df["inf_TKN"], df["ana_TKN"])
df["ana_NH4N_change_pct"] = removal_pct(df["inf_NH4N"], df["ana_NH4N"])
df["ana_MLSS_change_pct"] = removal_pct(df["inf_MLSS"], df["ana_MLSS"])
df["ana_MLVSS_change_pct"] = removal_pct(df["inf_MLVSS"], df["ana_MLVSS"])

# Stage deltas for pH and conductivity.
df["ana_minus_inf_pH"] = df["ana_pH"] - df["inf_pH"]
df["ana_minus_inf_cond"] = df["ana_cond"] - df["inf_cond"]

engineered_cols = [c for c in df.columns if c not in df_raw.columns and not c.startswith("target_")]
print(f"Engineered predictor columns: {len(engineered_cols)}")
engineered_cols


Engineered predictor columns: 18


['inf_CODs_to_CODt_ratio',
 'inf_MLVSS_to_MLSS_ratio',
 'inf_NH4N_to_TKN_ratio',
 'inf_CODt_to_TKN_ratio',
 'inf_CODs_to_TKN_ratio',
 'ana_CODs_to_CODt_ratio',
 'ana_MLVSS_to_MLSS_ratio',
 'ana_NH4N_to_TKN_ratio',
 'ana_CODt_to_TKN_ratio',
 'ana_CODs_to_TKN_ratio',
 'ana_CODt_removal_pct',
 'ana_CODs_removal_pct',
 'ana_TKN_change_pct',
 'ana_NH4N_change_pct',
 'ana_MLSS_change_pct',
 'ana_MLVSS_change_pct',
 'ana_minus_inf_pH',
 'ana_minus_inf_cond']

## 5. Remove Leakage and Unnecessary Columns


In [7]:
# These raw final effluent measurements are excluded from the ML feature table.
# They are either part of the target definition or measured after the prediction point.
leakage_cols = [c for c in df.columns if c.startswith("eff_")]

# Raw influent and anaerobic columns are also excluded here because this training table
# is intended to use feature-engineered process predictors only.
raw_upstream_cols = [c for c in df_raw.columns if c.startswith("inf_") or c.startswith("ana_")]

feature_cols = engineered_cols
ml_df = df[feature_cols + target_cols].copy()

print(f"Dropped leakage columns: {leakage_cols}")
print(f"Dropped raw upstream columns: {raw_upstream_cols}")
print(f"ML-ready shape: {ml_df.shape}")
ml_df.head()


Dropped leakage columns: ['eff_pH', 'eff_cond', 'eff_CODt', 'eff_CODs', 'eff_MLSS', 'eff_MLVSS', 'eff_TKN', 'eff_NH4N']
Dropped raw upstream columns: ['inf_pH', 'inf_cond', 'inf_CODt', 'inf_CODs', 'inf_MLSS', 'inf_MLVSS', 'inf_TKN', 'inf_NH4N', 'ana_pH', 'ana_cond', 'ana_CODt', 'ana_CODs', 'ana_MLSS', 'ana_MLVSS', 'ana_TKN', 'ana_NH4N']
ML-ready shape: (27, 20)


,inf_CODs_to_CODt_ratio,inf_MLVSS_to_MLSS_ratio,inf_NH4N_to_TKN_ratio,inf_CODt_to_TKN_ratio,inf_CODs_to_TKN_ratio,ana_CODs_to_CODt_ratio,ana_MLVSS_to_MLSS_ratio,ana_NH4N_to_TKN_ratio,ana_CODt_to_TKN_ratio,ana_CODs_to_TKN_ratio,ana_CODt_removal_pct,ana_CODs_removal_pct,ana_TKN_change_pct,ana_NH4N_change_pct,ana_MLSS_change_pct,ana_MLVSS_change_pct,ana_minus_inf_pH,ana_minus_inf_cond,target_final_CODt_removal_pct,target_final_CODs_removal_pct
0,0.80,0.90,0.22,24.06,19.21,0.23,0.93,0.71,4.62,1.06,75.81,93.03,-25.89,-300.00,-62.34,-69.20,NaN,NaN,93.32,93.26
1,0.91,0.84,0.19,18.50,16.86,0.31,0.70,0.63,6.28,1.92,79.07,92.96,38.37,-100.00,-147.06,-104.65,3.46,NaN,96.92,96.80
2,0.90,0.86,0.22,19.97,17.92,0.33,0.75,0.62,6.74,2.20,74.66,90.77,24.87,-109.09,-48.89,-29.87,2.72,NaN,97.10,97.03
3,0.94,0.85,0.88,21.64,20.32,0.52,0.82,0.73,3.97,2.08,74.52,85.76,-38.74,-14.29,-30.90,-26.90,2.56,NaN,94.38,95.65
4,0.57,0.67,0.23,46.77,26.75,0.38,0.69,0.69,6.31,2.38,68.59,79.32,-132.81,-586.67,-35.52,-40.00,NaN,NaN,96.02,95.04


## 6. Feature Missingness Review


In [8]:
missing_summary = (
    ml_df.drop(columns=target_cols)
    .isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: x["missing_count"] / len(ml_df) * 100)
    .sort_values("missing_pct", ascending=False)
)
missing_summary.head(20)


,missing_count,missing_pct
ana_minus_inf_cond,12,44.44
ana_minus_inf_pH,9,33.33
ana_MLVSS_change_pct,4,14.81
inf_MLVSS_to_MLSS_ratio,4,14.81
ana_MLVSS_to_MLSS_ratio,4,14.81
ana_MLSS_change_pct,4,14.81
ana_TKN_change_pct,3,11.11
ana_NH4N_to_TKN_ratio,3,11.11
ana_CODt_to_TKN_ratio,2,7.41
inf_NH4N_to_TKN_ratio,2,7.41


In [9]:
# Row-level missingness diagnostics. These are reviewed here but not added to the training CSV.
row_quality = pd.DataFrame({
    "has_complete_primary_targets": ml_df[target_cols].notna().all(axis=1),
    "row_missing_feature_count": ml_df[feature_cols].isna().sum(axis=1),
    "row_missing_feature_pct": ml_df[feature_cols].isna().mean(axis=1) * 100,
})

row_quality.head()


,has_complete_primary_targets,row_missing_feature_count,row_missing_feature_pct
0,True,2,11.11
1,True,1,5.56
2,True,1,5.56
3,True,1,5.56
4,True,2,11.11


## 7. Save ML-Ready CSV


In [10]:
ml_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Final ML-ready shape: {ml_df.shape}")


Saved: /home/itri/projects/water-quality-ml/water_quality_ml_ready.csv
Final ML-ready shape: (27, 20)


## 8. Data Dictionary


In [11]:
data_dictionary = pd.DataFrame({
    "column": ml_df.columns,
    "role": [
        "target" if c in target_cols else "engineered_feature"
        for c in ml_df.columns
    ],
    "missing_count": ml_df.isna().sum().values,
    "missing_pct": (ml_df.isna().mean() * 100).round(1).values,
})

data_dictionary


,column,role,missing_count,missing_pct
0,inf_CODs_to_CODt_ratio,engineered_feature,0,0.00
1,inf_MLVSS_to_MLSS_ratio,engineered_feature,4,14.80
2,inf_NH4N_to_TKN_ratio,engineered_feature,2,7.40
3,inf_CODt_to_TKN_ratio,engineered_feature,1,3.70
4,inf_CODs_to_TKN_ratio,engineered_feature,1,3.70
5,ana_CODs_to_CODt_ratio,engineered_feature,0,0.00
6,ana_MLVSS_to_MLSS_ratio,engineered_feature,4,14.80
7,ana_NH4N_to_TKN_ratio,engineered_feature,3,11.10
8,ana_CODt_to_TKN_ratio,engineered_feature,2,7.40
9,ana_CODs_to_TKN_ratio,engineered_feature,2,7.40


Notes for downstream modeling:

- Use `target_final_CODt_removal_pct` and/or `target_final_CODs_removal_pct` as the prediction targets.
- The exported CSV contains only feature-engineered process predictors and the two final COD removal targets.
- Raw `inf_*`, raw `ana_*`, raw `eff_*`, date/id, quality-check, and final-effluent concentration target columns are excluded from the exported training table.
- If you later want an influent-only early-prediction table, filter the predictors to engineered columns beginning with `inf_`.
